# Assignment #4 — Anggota 1: Baseline & Data-Quality Review + Kerangka Bersama

**Satu notebook ini berisi semuanya**: kerangka bersama (split, CV, metrik, hook), baseline, dan review masalah data.
Tidak ada file `.py` terpisah.

## Cara pakai (ringkas)
1. Letakkan `heart_disease_cleveland.csv` di `./dataset/` (path sama dengan notebook Assignment #3).
2. Install library (lihat sel 1), lalu jalankan notebook **dari atas ke bawah** (`Run All`). Total ±5–10 menit di CPU
   (dua eksperimen ensemble × 15 fold × ±2 menit; sel baseline B1 yang paling lama).
3. Jangan mengubah **Bagian A** (kerangka) setelah anggota lain mulai memakainya — split & fold sudah dikunci di `./artifacts/`.
4. Bagikan folder `./artifacts/` (khususnya `split.json` dan `cv_splits.json`) ke Anggota 2, 3, 4 agar fold **identik**.

## Peta notebook
| Bagian | Isi | Instruksi soal |
|---|---|---|
| **A. Kerangka bersama** | setup, load data, split terkunci, fold CV terkunci, preprocessing E0/E5, metrik, model ensemble, `evaluate_cv` dengan hook | fondasi Instruksi 1, 3, 4, 5, 6, 8 |
| **B. Baseline** | CV E0 vs E5, pilih & kunci baseline, simpan OOF | Instruksi 1 |
| **C. Data-quality review** | bukti kuantitatif masalah data (hanya data train) + tabel masalah | Instruksi 2 |
| **D. Serah-terima & evaluasi akhir** | `final_test_eval` (jangan dijalankan dulu), instruksi reproduksi | Instruksi 8, Submission |

# BAGIAN A — Kerangka bersama (jangan diubah setelah dibagikan)

## A1. Setup & konfigurasi
Semua konstanta ada di satu sel agar mudah dilacak. `N_SPLITS × N_REPEATS` menentukan jumlah fold:
5 × 3 = 15 fold. Fold validasi ±49 baris, jadi hasil per fold tetap berisik (std ±0,05); itu normal untuk dataset 242 baris.

Library yang dibutuhkan (jalankan sekali di terminal/sel, lalu restart kernel):
```
pip install numpy pandas scikit-learn tensorflow imbalanced-learn cleanlab shap lime
```

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import pandas as pd
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, matthews_corrcoef, roc_auc_score,
                             average_precision_score, confusion_matrix)

# ---------------- KONFIGURASI (ubah HANYA sekali di awal, lalu hapus ./artifacts/*.json) ----------------
PATH       = "./dataset/heart_disease_cleveland.csv"
ART_DIR    = "./artifacts"
SEED       = 42
TARGET     = "num"
N_SPLITS   = 5
N_REPEATS  = 3
TEST_SIZE  = 0.2
RF_PARAMS  = {"n_estimators": 194, "max_depth": 43}                      # dari Assignment #3
CNN_PARAMS = {"filters": 34, "kernel_size": 2, "dense_units": 50,
              "learning_rate": 0.001, "epochs": 50, "batch_size": 32}     # dari Assignment #3
os.makedirs(ART_DIR, exist_ok=True)

print("sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)
try:
    import tensorflow as tf; print("tensorflow", tf.__version__)
    import logging; logging.getLogger("tensorflow").setLevel(logging.ERROR)   # sembunyikan peringatan retracing yang tidak berbahaya
except Exception as e:
    print("TensorFlow belum terpasang:", e)

## A2. Load data + pemeriksaan integritas (boleh pada data penuh)
EDA Guideline mengizinkan **sebelum split** hanya pemeriksaan struktur & kualitas: jumlah baris/kolom, tipe, missing value,
duplikat, definisi target, nilai mustahil. **Tidak** ada analisis hubungan fitur–target di sini.

`row_id` = nomor baris asli CSV. Ia menjadi kunci stabil untuk tabel keputusan Cleanlab (Anggota 3) dan koreksi label.

In [ ]:
df = pd.read_csv(PATH).rename(columns={"target": TARGET})
df.insert(0, "row_id", np.arange(len(df)))

print("Bentuk:", df.shape)
print("Missing per kolom (data penuh):", df.isna().sum()[df.isna().sum() > 0].to_dict())
print("Duplikat persis:", df.drop(columns="row_id").duplicated().sum())
print("Distribusi target (data penuh):", df[TARGET].value_counts().sort_index().to_dict())
assert set(df[TARGET].unique()) == {0, 1}, "target harus biner 0/1"
df.head(3)

## A3. Split train/test — dibuat SEKALI, lalu dikunci ke file
Split stratified 80/20 seed 42 (sama dengan Assignment #3). Setelah `split.json` ada, split **dibaca dari file**, bukan dibuat ulang,
sehingga tidak bisa bergeser walau versi library berbeda. Test set **tidak boleh disentuh** sampai Instruksi 8.

In [ ]:
def get_split(df):
    p = f"{ART_DIR}/split.json"
    if os.path.exists(p):
        ids = json.load(open(p))
    else:
        tr, te = train_test_split(df, test_size=TEST_SIZE, random_state=SEED, stratify=df[TARGET])
        ids = {"train": sorted(tr.row_id.tolist()), "test": sorted(te.row_id.tolist())}
        json.dump(ids, open(p, "w"))
    train_df = df[df.row_id.isin(ids["train"])].reset_index(drop=True)
    test_df  = df[df.row_id.isin(ids["test"])].reset_index(drop=True)
    assert not set(train_df.row_id) & set(test_df.row_id), "train/test overlap!"
    return train_df, test_df

train_df, test_df = get_split(df)
print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Kelas train:", train_df[TARGET].value_counts().sort_index().to_dict())
# Dari sini ke bawah, test_df HANYA dipakai oleh final_test_eval() di Bagian D.

## A4. Fold cross-validation — dibuat sekali, disimpan berbasis `row_id`
`RepeatedStratifiedKFold` (5 fold × 3 repeat). Semua anggota memakai fold yang sama, sehingga selisih antar-eksperimen dapat
dibandingkan **berpasangan per fold** (lebih tajam daripada membandingkan mean ± std saja).

In [ ]:
def get_cv_splits(train_df):
    p = f"{ART_DIR}/cv_splits.json"
    if os.path.exists(p):
        return [(np.array(a), np.array(b), r) for a, b, r in json.load(open(p))]
    rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
    out = []
    for i, (a, b) in enumerate(rskf.split(train_df, train_df[TARGET])):
        out.append((train_df.row_id.values[a].tolist(), train_df.row_id.values[b].tolist(), i // N_SPLITS))
    json.dump(out, open(p, "w"))
    return [(np.array(a), np.array(b), r) for a, b, r in out]

cv = get_cv_splits(train_df)
print("Jumlah fold:", len(cv), "| ukuran validasi fold-0:", len(cv[0][1]))

## A5. Preprocessing E0 dan E5 (disalin apa adanya dari notebook Assignment #3)
- **E0** = baseline Assignment #2 (imputasi + hapus outlier IQR di train + Min-Max).
- **E5** = pipeline terbaik hasil seleksi notebook #3 (one-hot, clipping, log1p, rekayasa fitur `hr_reserve` & `st_severity`, flag missing).
- Kontrak: `f(tr_df, ev_df) -> (X_tr, y_tr, X_ev, feature_names)`. Semua statistik (median, kuantil, scaler, encoder) di-**fit hanya di `tr_df`**,
  jadi aman dipanggil di dalam fold CV (syarat Instruksi 4).
- Pipeline `current` (seleksi GA+CSO) **sengaja tidak dipakai**: fitur terpilihnya dihitung dari seluruh data train sehingga bocor ke fold validasi.
- E1–E4 tidak disertakan (tidak dibutuhkan untuk baseline; E5 berdiri sendiri).

In [ ]:
# Konstanta kolom (dari notebook Assignment #3)
continuous_cols   = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_cols  = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
NOMINAL_COLS      = ['cp', 'restecg', 'thal']
ORDINAL_COLS      = ['ca', 'slope']
BINARY_COLS       = ['sex', 'fbs', 'exang']
LOG_COLS          = ['chol', 'oldpeak']
MISSING_FLAG_COLS = ['ca', 'thal']
ENGINEERED_COLS   = ['hr_reserve', 'st_severity']


class IQRClipper(BaseEstimator, TransformerMixin):
    """Winsorize dengan batas IQR (baris TIDAK dihapus). Batas dihitung dari data fit (train saja)."""
    def __init__(self, k=1.5):
        self.k = k
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, 25, axis=0), np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lo_, self.hi_ = q1 - self.k * iqr, q3 + self.k * iqr
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lo_, self.hi_)


def _base_impute(tr, ev):
    """Imputasi median (kontinu) / modus (kategorikal). Fit HANYA di train, transform ke train & eval."""
    num_imp = SimpleImputer(strategy='median').fit(tr[continuous_cols])
    cat_imp = SimpleImputer(strategy='most_frequent').fit(tr[categorical_cols])
    for d in (tr, ev):
        d[continuous_cols] = num_imp.transform(d[continuous_cols])
        d[categorical_cols] = cat_imp.transform(d[categorical_cols])


def preprocess_E0(tr_df, ev_df):
    """Baseline Assignment #2: imputasi + IQR outlier REMOVAL (drop baris TRAIN saja) + Min-Max
    di semua fitur mentah. Belum ada one-hot, flag missing, rekayasa fitur, atau log transform."""
    tr, ev = tr_df.copy(), ev_df.copy()
    y_tr = tr.pop(TARGET).to_numpy(dtype=int)

    _base_impute(tr, ev)
    for c in categorical_cols:
        tr[c] = tr[c].round().astype(int)
        ev[c] = ev[c].round().astype(int)

    q1, q3 = tr[continuous_cols].quantile(0.25), tr[continuous_cols].quantile(0.75)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    mask = ((tr[continuous_cols] >= lo) & (tr[continuous_cols] <= hi)).all(axis=1)
    y_tr = y_tr[mask.to_numpy()]
    tr = tr.loc[mask].reset_index(drop=True)

    feature_names = continuous_cols + categorical_cols
    scaler = MinMaxScaler().fit(tr[feature_names])
    X_train = scaler.transform(tr[feature_names])
    X_eval = scaler.transform(ev[feature_names])
    return X_train, y_tr, X_eval, feature_names


def preprocess_E5(tr_df, ev_df):
    """E4 + flag missing (_missing) untuk ca & thal (checklist poin 1).
    Setara dengan preprocess_current_notebook TAPI TANPA restriksi ke selected_feature_names
    (seleksi fitur GA+CSO sengaja dipisah -- checklist poin 6, 'dilakukan di Feature Extraction')."""
    tr, ev = tr_df.copy(), ev_df.copy()
    y_tr = tr.pop(TARGET).to_numpy(dtype=int)

    missing_flag_cols = [f'{c}_missing' for c in MISSING_FLAG_COLS]
    for d in (tr, ev):
        for c in MISSING_FLAG_COLS:
            d[f'{c}_missing'] = d[c].isna().astype(int)

    _base_impute(tr, ev)
    for c in NOMINAL_COLS + BINARY_COLS + ORDINAL_COLS:
        tr[c] = tr[c].round().astype(int)
        ev[c] = ev[c].round().astype(int)

    for d in (tr, ev):
        d['hr_reserve'] = d['thalach'] - (220 - d['age'])
        d['st_severity'] = d['oldpeak'] * d['slope']

    for c in LOG_COLS:
        tr[c] = np.log1p(tr[c])
        ev[c] = np.log1p(ev[c])

    clip_cols = continuous_cols + ENGINEERED_COLS
    clipper = IQRClipper().fit(tr[clip_cols])
    tr[clip_cols] = clipper.transform(tr[clip_cols])
    ev[clip_cols] = clipper.transform(ev[clip_cols])

    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).fit(tr[NOMINAL_COLS].astype(str))
    nom_names = list(ohe.get_feature_names_out(NOMINAL_COLS))
    scale_cols = continuous_cols + ENGINEERED_COLS + ORDINAL_COLS
    scaler = MinMaxScaler().fit(tr[scale_cols])

    def assemble(d):
        num = pd.DataFrame(scaler.transform(d[scale_cols]), columns=scale_cols, index=d.index)
        nom = pd.DataFrame(ohe.transform(d[NOMINAL_COLS].astype(str)), columns=nom_names, index=d.index)
        rest = d[BINARY_COLS + missing_flag_cols]
        return pd.concat([num, rest, nom], axis=1)

    feature_names = scale_cols + BINARY_COLS + missing_flag_cols + nom_names
    return (assemble(tr)[feature_names].to_numpy(dtype=float), y_tr,
            assemble(ev)[feature_names].to_numpy(dtype=float), feature_names)


PREPROCESS = {'E0': preprocess_E0, 'E5': preprocess_E5}

## A6. Metrik evaluasi
Soal meminta metrik yang sesuai untuk data imbalanced (bukan hanya accuracy). Set metrik ini dipakai **semua** eksperimen:
accuracy, **macro F1** (metrik utama), balanced accuracy, **MCC**, precision/recall/F1 kelas positif, specificity, ROC-AUC, **PR-AUC**.

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy":     accuracy_score(y_true, y_pred),
        "macro_f1":     f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "mcc":          matthews_corrcoef(y_true, y_pred),
        "precision":    precision_score(y_true, y_pred, zero_division=0),
        "recall":       recall_score(y_true, y_pred, zero_division=0),
        "f1_pos":       f1_score(y_true, y_pred, zero_division=0),
        "specificity":  tn / (tn + fp) if (tn + fp) else np.nan,
        "roc_auc":      roc_auc_score(y_true, y_prob),
        "pr_auc":       average_precision_score(y_true, y_prob),
    }
METRIC_COLS = ["accuracy", "macro_f1", "balanced_acc", "mcc", "roc_auc", "pr_auc"]

## A7. Model: ensemble RF + CNN (sama dengan Assignment #3)
RF dan CNN 1D dilatih pada fold yang sama; prediksi akhir = *majority vote*, dengan rata-rata probabilitas sebagai pemutus saat kedua model berbeda.
Probabilitas ensemble = rata-rata probabilitas RF dan CNN (dipakai untuk AUC dan Cleanlab).

- `keep_models=False` (default, untuk loop CV): sesi TensorFlow dibersihkan tiap fold agar memori tidak bocor.
- `keep_models=True` (untuk model final, dipakai **Anggota 4**): hasil memuat `predict_fn(X)` yang bisa langsung dimasukkan ke `shap.KernelExplainer` / LIME.
- `rf_fit_predict` = RF saja, cepat; cocok untuk percobaan cepat atau `shap.TreeExplainer` (sebutkan sebagai limitasi bila dipakai).

In [ ]:
def rf_fit_predict(X_tr, y_tr, X_ev, sample_weight=None, **_):
    rf = RandomForestClassifier(random_state=SEED, **RF_PARAMS).fit(X_tr, y_tr, sample_weight=sample_weight)
    return {"prob": rf.predict_proba(X_ev)[:, 1], "model": rf}


def make_ensemble_fit_predict(rf_params=None, cnn_params=None, keep_models=False):
    rf_params, cnn_params = rf_params or RF_PARAMS, cnn_params or CNN_PARAMS

    def fit_predict(X_tr, y_tr, X_ev, sample_weight=None, **_):
        import tensorflow as tf
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
        from tensorflow.keras.optimizers import Adam
        rf = RandomForestClassifier(random_state=SEED, **rf_params).fit(X_tr, y_tr, sample_weight=sample_weight)
        p_rf = rf.predict_proba(X_ev)[:, 1]
        tf.keras.utils.set_random_seed(SEED)
        m = Sequential([Conv1D(cnn_params["filters"], cnn_params["kernel_size"], activation="relu",
                               input_shape=(X_tr.shape[1], 1)),
                        MaxPooling1D(2), Flatten(),
                        Dense(cnn_params["dense_units"], activation="relu"),
                        Dense(1, activation="sigmoid")])
        m.compile(optimizer=Adam(cnn_params["learning_rate"]), loss="binary_crossentropy")
        m.fit(X_tr[..., None], y_tr, sample_weight=sample_weight, epochs=cnn_params["epochs"],
              batch_size=cnn_params["batch_size"], validation_split=0.2, verbose=0)
        p_cnn = m.predict(X_ev[..., None], verbose=0).ravel()
        votes = (p_rf >= .5).astype(int) + (p_cnn >= .5).astype(int)
        avg   = .5 * p_rf + .5 * p_cnn
        pred  = np.where(votes == 2, 1, np.where(votes == 0, 0, (avg >= .5).astype(int)))
        res = {"prob": avg, "pred": pred, "rf_prob": p_rf, "cnn_prob": p_cnn, "model": rf}
        if keep_models:
            res["cnn_model"] = m
            res["predict_fn"] = lambda X: .5 * rf.predict_proba(np.asarray(X, float))[:, 1] + \
                                          .5 * m.predict(np.asarray(X, float)[..., None], verbose=0).ravel()
        else:
            tf.keras.backend.clear_session()
        return res
    return fit_predict

## A8. `evaluate_cv` — harness CV bersama (berisi hook untuk Anggota 2, 3, 4)
Aturan yang dikunci di fungsi ini:
1. **Preprocessing dijalankan di dalam setiap fold** (fit di bagian train fold saja).
2. Modifikasi dataset (`label_overrides`, `drop_ids`, `resampler`, bobot kelas) **hanya diterapkan ke bagian train fold**.
   Fold validasi selalu memuat **semua barisnya**.
3. Secara default validasi memakai **label asli** (`eval_on_corrected=False`). Jika koreksi label benar, metrik justru menganggapnya "salah";
   karena itu laporkan juga `eval_on_corrected=True` untuk D2/D3 dan tulis keduanya di laporan.
4. Hasil per fold disimpan ke `./artifacts/cv_results.csv` (satu baris per fold per eksperimen), dan fungsi mengembalikan probabilitas
   **out-of-fold** (rata-rata antar-repeat) yang dipakai Cleanlab.

| Hook | Dipakai oleh | Contoh |
|---|---|---|
| `resampler(X, y) -> (X, y)` | Anggota 2 | `lambda X, y: SMOTE(random_state=42).fit_resample(X, y)` |
| `sample_weight_fn(y) -> bobot` | Anggota 2 | `lambda y: compute_sample_weight("balanced", y)` |
| `label_overrides={row_id: label}` , `drop_ids=[row_id,...]` | Anggota 3 | hanya baris dengan keputusan Correct / Remove |
| `fit_predict=make_ensemble_fit_predict(keep_models=True)` (di luar CV) | Anggota 4 | SHAP / LIME |

In [ ]:
def apply_train_modifications(tr, label_overrides=None, drop_ids=None):
    tr = tr.copy()
    if label_overrides:
        m = tr.row_id.isin(label_overrides)
        tr.loc[m, TARGET] = tr.loc[m, "row_id"].map(label_overrides).astype(int)
    if drop_ids:
        tr = tr[~tr.row_id.isin(set(drop_ids))]
    return tr.reset_index(drop=True)


def evaluate_cv(exp_code, train_df, preprocess_fn, fit_predict=None,
                label_overrides=None, drop_ids=None, resampler=None, sample_weight_fn=None,
                eval_on_corrected=False, results_csv=f"{ART_DIR}/cv_results.csv", verbose=True):
    fit_predict = fit_predict or make_ensemble_fit_predict()
    rows, oof = [], []
    by_id = train_df.set_index("row_id", drop=False)
    for k, (tr_ids, va_ids, rep) in enumerate(get_cv_splits(train_df)):
        tr = apply_train_modifications(by_id.loc[tr_ids].reset_index(drop=True), label_overrides, drop_ids)
        va = by_id.loc[va_ids].reset_index(drop=True)
        X_tr, y_tr, X_ev, feats = preprocess_fn(tr.drop(columns=["row_id"]),
                                                va.drop(columns=["row_id", TARGET]))
        X_tr, X_ev, y_tr = np.asarray(X_tr, float), np.asarray(X_ev, float), np.asarray(y_tr, int)
        assert len(X_ev) == len(va), "preprocessing tidak boleh membuang baris validasi"
        sw = sample_weight_fn(y_tr) if sample_weight_fn else None
        if resampler:
            X_tr, y_tr = resampler(X_tr, y_tr); sw = None
        out  = fit_predict(X_tr, y_tr, X_ev, sample_weight=sw)
        prob = out["prob"]
        pred = out.get("pred", (prob >= .5).astype(int))
        if eval_on_corrected and label_overrides:
            va = apply_train_modifications(va, label_overrides, None)
        y_va = va[TARGET].to_numpy(int)
        rows.append({"exp": exp_code, "fold": k, "repeat": rep, "n_train": len(y_tr),
                     **compute_metrics(y_va, pred, prob)})
        oof.append(pd.DataFrame({"row_id": va.row_id, "repeat": rep, "prob": prob, "y": y_va}))
    fold_df = pd.DataFrame(rows)
    oof_df = pd.concat(oof).groupby("row_id").agg(prob=("prob", "mean"), y=("y", "first")).reset_index()

    old = pd.read_csv(results_csv) if os.path.exists(results_csv) else pd.DataFrame()
    pd.concat([old[old.exp != exp_code] if len(old) else old, fold_df]).to_csv(results_csv, index=False)
    if verbose:
        print(f"[{exp_code}] {len(fold_df)} fold  |  mean ± std")
        s = fold_df[METRIC_COLS].agg(["mean", "std"]).T
        print("\n".join(f"  {m:13s} {r['mean']:.4f} ± {r['std']:.4f}" for m, r in s.iterrows()))
    return fold_df, oof_df


def summarize(results_csv=f"{ART_DIR}/cv_results.csv", metric="macro_f1"):
    # Tabel perbandingan mean ± std seluruh eksperimen (untuk laporan).
    r = pd.read_csv(results_csv)
    g = r.groupby("exp")[METRIC_COLS].agg(["mean", "std"])
    return g.sort_values((metric, "mean"), ascending=False).round(4)


def paired_delta(baseline, other, metric="macro_f1", results_csv=f"{ART_DIR}/cv_results.csv"):
    # Selisih berpasangan per fold (fold identik): mean, std, jumlah fold menang/kalah/seri.
    r = pd.read_csv(results_csv).pivot(index="fold", columns="exp", values=metric)
    d = r[other] - r[baseline]
    return {"delta_mean": round(d.mean(), 4), "delta_std": round(d.std(), 4),
            "menang": int((d > 0).sum()), "kalah": int((d < 0).sum()), "seri": int((d == 0).sum())}

# BAGIAN B — Baseline (Instruksi 1)

## B1. Jalankan CV untuk E0 dan E5
Soal: *"Use the best pipeline from Assignment #3 as the baseline … Report the baseline performance using appropriate evaluation metrics."*

Kita jalankan kedua kandidat dengan **model, fold, dan metrik yang sama**. Sel ini yang paling lama (±2 menit per eksperimen).
Hasil per fold otomatis tersimpan di `./artifacts/cv_results.csv`.

In [ ]:
ens = make_ensemble_fit_predict()
t0 = time.time()
fold_E0, oof_E0 = evaluate_cv("E0", train_df, PREPROCESS["E0"], ens)
print(f"  -> {time.time()-t0:.0f} detik\n")
t0 = time.time()
fold_E5, oof_E5 = evaluate_cv("E5", train_df, PREPROCESS["E5"], ens)
print(f"  -> {time.time()-t0:.0f} detik")

## B2. Bandingkan dan kunci baseline
Tampilkan tabel dan selisih berpasangan per fold. **Cara membaca:** bila `|delta_mean|` jauh lebih kecil dari `delta_std`
(dan menang/kalah tidak timpang), kedua pipeline **setara secara statistik**. Dari run uji saya E5 − E0 ≈ −0,012 dengan std ≈ 0,025 (E5 menang di 4 fold, kalah di 9, seri 2),
artinya selisih lebih kecil dari noise dan tidak ada pemenang yang jelas.

**Keputusan baseline** (isi `BASELINE_CODE`): soal meminta *"best pipeline from Assignment #3"*. Notebook #3 memilih **E5** sebagai
pipeline preprocessing terbaik selain E0, jadi default = `E5`. Tulis di laporan bahwa E0 setara di CV. Setelah dikunci, jangan diubah.

In [ ]:
display(summarize())
print("Selisih berpasangan E5 - E0 (macro F1):", paired_delta("E0", "E5"))

BASELINE_CODE = "E5"          # <- keputusan kelompok; kunci di sini
baseline_oof  = {"E0": oof_E0, "E5": oof_E5}[BASELINE_CODE]
baseline_oof.to_csv(f"{ART_DIR}/oof_baseline.csv", index=False)       # dipakai Cleanlab (Anggota 3)
print("Baseline terkunci:", BASELINE_CODE)

## B3. Tabel baseline untuk laporan
Satu baris dengan **mean ± std** seluruh metrik. Salin ke tabel perbandingan D0–D3 (kolom "CV Macro F1" dsb.).

In [ ]:
r = pd.read_csv(f"{ART_DIR}/cv_results.csv")
b = r[r.exp == BASELINE_CODE][METRIC_COLS + ["precision", "recall", "specificity"]].agg(["mean", "std"]).T
baseline_table = pd.DataFrame({"Baseline D0 (" + BASELINE_CODE + ")": b.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)})
baseline_table.to_csv(f"{ART_DIR}/baseline_table.csv")
baseline_table

# BAGIAN C — Data-quality review (Instruksi 2)

Soal: *"Do not repeat the complete EDA. Identify the unresolved data-quality problems … If a problem was already adequately addressed … do not investigate it again."*

Maka sel-sel berikut **hanya mengumpulkan bukti singkat** untuk tiap kandidat masalah, **hanya dari data train**
(sesuai EDA Guideline: analisis yang memengaruhi keputusan modeling tidak boleh memakai test set).

## C1. Class imbalance

In [ ]:
vc = train_df[TARGET].value_counts().sort_index()
print("Kelas train:", vc.to_dict(), "| rasio mayoritas:minoritas = %.2f" % (vc.max() / vc.min()))

# Apakah model memperlakukan kedua kelas sama baik? (dari prediksi out-of-fold baseline)
oof = baseline_oof.copy()
oof["pred"] = (oof.prob >= .5).astype(int)
print("\nConfusion matrix OOF baseline (baris = asli, kolom = prediksi):")
print(pd.crosstab(oof.y, oof.pred))
print("\nRecall per kelas:", {int(k): round(float((d.pred == k).mean()), 3) for k, d in oof.groupby("y")})

**Cara membaca:** rasio kelas hanya ≈ 1,18 (ringan), **tetapi** prediksi out-of-fold tidak simetris: pada run uji saya recall kelas 0 (sehat) ≈ 0,88
sedangkan kelas 1 (penyakit) ≈ 0,73, yaitu 30 *false negative* berbanding 16 *false positive*. Model cenderung "kalah" di kelas minoritas, dan salah-prediksi
penyakit adalah kesalahan yang lebih mahal secara klinis. Jadi imbalance **layak diuji** (class weighting dan SMOTE), dengan keputusan berdasarkan macro F1
dan recall kelas 1 — bukan diasumsikan tidak berlaku. Soal juga menyatakan perlakuan tidak perlu dipakai bila distribusi cukup seimbang;
bila hasilnya tidak membaik, itu tetap hasil yang sah (Anggota 2 menjelaskannya).

## C2. Kategori langka dan flag missing (risiko fitur yang tidak stabil)

In [ ]:
print("Missing pada data penuh :", df[["ca", "thal"]].isna().sum().to_dict())
print("Missing pada TRAIN      :", train_df[["ca", "thal"]].isna().sum().to_dict())
for c in ["restecg", "cp", "thal", "ca", "slope"]:
    print(f"{c:8s} (train):", train_df[c].value_counts(dropna=False).sort_index().to_dict())

**Cara membaca:** bila `ca`/`thal` hanya kosong pada 1–2 baris train, maka fitur `ca_missing`/`thal_missing` di E5 hanya "melihat" 1–2 observasi;
begitu juga `restecg_1` jika kategori `1` hanya muncul 1 kali. Model/seleksi fitur yang menganggap fitur seperti ini penting kemungkinan
*menghafal baris tunggal*, bukan menemukan pola. Ini temuan untuk tabel masalah, dan untuk Anggota 4 saat membaca SHAP.

## C3. Fitur rekayasa yang redundan

In [ ]:
t = train_df.copy()
t["hr_reserve"] = t.thalach - (220 - t.age)
from numpy.linalg import lstsq
A = np.c_[t.thalach, t.age, np.ones(len(t))]
coef, res, *_ = lstsq(A, t.hr_reserve, rcond=None)
r2 = 1 - ((t.hr_reserve - A @ coef) ** 2).sum() / ((t.hr_reserve - t.hr_reserve.mean()) ** 2).sum()
print("R² memprediksi hr_reserve dari (thalach, age) =", round(r2, 6), "-> kombinasi linear persis")
print("corr(hr_reserve, thalach) =", round(t.hr_reserve.corr(t.thalach), 3))

**Cara membaca:** R² = 1 berarti `hr_reserve` tidak membawa informasi baru selain `thalach` dan `age`; ia hanya menggandakan mereka.
Pada SHAP, kontribusi `thalach`/`age` bisa terpecah ke `hr_reserve` sehingga pembacaan importance menyesatkan (catatan untuk Anggota 4).

## C4. Outlier (sudah ditangani di #3 — hanya hitung sebagai konteks)

In [ ]:
q1, q3 = train_df[continuous_cols].quantile(.25), train_df[continuous_cols].quantile(.75)
iqr = q3 - q1
out_mask = (train_df[continuous_cols] < q1 - 1.5 * iqr) | (train_df[continuous_cols] > q3 + 1.5 * iqr)
print("Jumlah outlier IQR per kolom (train):", out_mask.sum().to_dict())
print("Baris dengan >=1 outlier           :", int(out_mask.any(axis=1).sum()), "dari", len(train_df),
      "(E0 membuang baris ini; E5 hanya meng-clip-nya)")

## C5. Duplikat dan near-duplicate; pasangan hampir identik dengan label berbeda
Duplikat persis sudah dicek di A2 (0). Di sini dicari **pasangan sangat mirip tetapi labelnya berbeda** — kandidat label bermasalah yang
relevan untuk Cleanlab (Anggota 3). Hanya data train; hanya untuk menjelaskan masalah, bukan untuk mengubah label.

In [ ]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
t = train_df.copy()
t[["ca", "thal"]] = t[["ca", "thal"]].fillna(t[["ca", "thal"]].mode().iloc[0])
Z = StandardScaler().fit_transform(t[continuous_cols + categorical_cols])
nn = NearestNeighbors(n_neighbors=2).fit(Z)
d, ix = nn.kneighbors(Z)
nn_df = pd.DataFrame({"row_id": t.row_id, "tetangga": t.row_id.values[ix[:, 1]], "jarak": d[:, 1],
                      "label": t[TARGET], "label_tetangga": t[TARGET].values[ix[:, 1]]})
close = nn_df.nsmallest(10, "jarak")
display(close.round(3))
print("Dari 10 pasangan terdekat, label berbeda:", int((close.label != close.label_tetangga).sum()))

**Cara membaca:** pasangan dengan jarak sangat kecil tetapi label berbeda = kandidat konflik label. Pada data ini jarak terdekat masih ≈ 0,85 (ruang terstandarisasi) dan semua label sama, jadi tidak ada near-duplicate. Daftar `row_id` di sini boleh diberikan ke Anggota 3
sebagai bukti tambahan, tetapi **bukan alasan koreksi** (soal: perbedaan model/tetangga saja bukan bukti cukup).

## C6. Tabel masalah data (untuk laporan Instruksi 2)
Tabel ini sudah mengikuti format soal. Angka di kolom "Bukti" mengacu ke output sel C1–C5; **cek ulang dengan output Anda sebelum menyalin ke laporan.**

| # | Masalah | Bukti | Status setelah #2/#3 | Ditangani di |
|---|---|---|---|---|
| 1 | Class imbalance ringan dengan error asimetris | rasio ±1,18, tetapi recall kelas 1 ≈ 0,73 vs kelas 0 ≈ 0,88 (C1) | Belum diuji di #2/#3 | Anggota 2 (Instr. 3) |
| 2 | Label mencurigakan / target hasil binarisasi 0–4 | belum pernah diperiksa di #2/#3; kasus batas (nilai asli 1) rawan noisy | **Belum terselesaikan** | Anggota 3 (Instr. 4–5) |
| 3 | Flag `ca_missing`, `thal_missing`, kategori `restecg=1` hanya didukung 1–2 baris train | C2 | **Belum terselesaikan**; tetap masuk E5 | Anggota 4 (Instr. 6–7) |
| 4 | `hr_reserve` redundan (R² = 1 dari `thalach` & `age`) | C3 | Belum terselesaikan; memengaruhi pembacaan SHAP | Anggota 4 |
| 5 | Near-duplicate / pasangan mirip berlabel beda | C5: 10 pasangan terdekat (jarak ≥ 0,85) semuanya berlabel sama | Tidak ada bukti masalah; tidak diselidiki lebih lanjut | — |
| 6 | Outlier (`chol`, `oldpeak`, dll.) | C4 | Sudah ditangani (clipping di E5); tidak diselidiki ulang | — |
| 7 | Duplikat persis | A2 = 0 | Tidak ada | — |
| 8 | Dataset kecil (242 train, 61 test): std antar-fold ±0,05, selisih antar-pipeline di bawah noise | B2 | Keterbatasan, bukan perbaikan | Dibahas di Instr. 9 |
| 9 | **Masalah proses** (bukan data): seleksi fitur GA+CSO memakai seluruh train (bocor ke fold CV), EDA fitur–target pada data penuh sebelum split, test set dievaluasi berulang di #3 | notebook #3 sel 16–19, 44, 55–66 | Dikoreksi di kerangka ini: preprocessing di dalam fold, test hanya lewat `final_test_eval` | Anggota 1 |

Sesuai Data_Preprocessing.pdf, setiap keputusan harus didokumentasikan beserta **rasional**. Kolom "Status" dan "Ditangani di" di atas adalah rasionalnya.

# BAGIAN D — Serah-terima, evaluasi akhir, dan reproduksi

## D1. Yang diserahkan ke anggota lain
Jalankan sel ini untuk memastikan semua artefak ada, lalu kirim folder `./artifacts/` beserta notebook ini.

In [ ]:
needed = ["split.json", "cv_splits.json", "cv_results.csv", "oof_baseline.csv", "baseline_table.csv"]
for f in needed:
    print(("OK   " if os.path.exists(f"{ART_DIR}/{f}") else "HILANG"), f)

## D2. Evaluasi akhir di test set (Instruksi 8) — **JANGAN dijalankan sekarang**
Fungsi ini akan dipanggil oleh Anggota 1 + 2 setelah dataset final diputuskan. Pagar di dalamnya menolak evaluasi ulang dengan `label` yang sama,
agar test set tidak dipakai untuk tuning. Pemanggilan nanti: sekali untuk baseline (D0) dan sekali untuk dataset final.

In [ ]:
def final_test_eval(label, train_df, test_df, preprocess_fn, fit_predict=None,
                    label_overrides=None, drop_ids=None, resampler=None, sample_weight_fn=None,
                    log_path=f"{ART_DIR}/test_eval_log.csv"):
    fit_predict = fit_predict or make_ensemble_fit_predict()
    log = pd.read_csv(log_path) if os.path.exists(log_path) else pd.DataFrame(columns=["label"])
    if label in set(log["label"]):
        raise RuntimeError(f"'{label}' sudah pernah dievaluasi di test set. Jangan diulang.")
    tr = apply_train_modifications(train_df, label_overrides, drop_ids)
    X_tr, y_tr, X_ev, _ = preprocess_fn(tr.drop(columns=["row_id"]), test_df.drop(columns=["row_id", TARGET]))
    X_tr, X_ev, y_tr = np.asarray(X_tr, float), np.asarray(X_ev, float), np.asarray(y_tr, int)
    sw = sample_weight_fn(y_tr) if sample_weight_fn else None
    if resampler:
        X_tr, y_tr = resampler(X_tr, y_tr); sw = None
    out  = fit_predict(X_tr, y_tr, X_ev, sample_weight=sw)
    pred = out.get("pred", (out["prob"] >= .5).astype(int))
    m = {"label": label, **compute_metrics(test_df[TARGET].to_numpy(int), pred, out["prob"])}
    pd.concat([log, pd.DataFrame([m])]).to_csv(log_path, index=False)
    return m

# --- DIJALANKAN NANTI (Instruksi 8), bukan sekarang: ---
# m_base  = final_test_eval("D0_baseline", train_df, test_df, PREPROCESS[BASELINE_CODE])
# m_final = final_test_eval("D_final",     train_df, test_df, PREPROCESS[BASELINE_CODE],
#                           label_overrides=..., drop_ids=..., resampler=...)   # sesuai keputusan Anggota 2 & 3

## D3. Contoh pemakaian hook oleh anggota lain (jangan dijalankan di sini)
```python
# Anggota 2 — class weighting dan SMOTE (hanya di dalam fold)
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE
evaluate_cv("D1_cw",    train_df, PREPROCESS[BASELINE_CODE], sample_weight_fn=lambda y: compute_sample_weight("balanced", y))
evaluate_cv("D1_smote", train_df, PREPROCESS[BASELINE_CODE], resampler=lambda X, y: SMOTE(random_state=SEED).fit_resample(X, y))

# Anggota 3 — Cleanlab memakai probabilitas out-of-fold baseline
from cleanlab.filter import find_label_issues
from cleanlab.rank import get_label_quality_scores
oof = pd.read_csv(f"{ART_DIR}/oof_baseline.csv")
P   = np.c_[1 - oof.prob, oof.prob]
issues = find_label_issues(oof.y.values, P, return_indices_ranked_by="self_confidence")
quality = get_label_quality_scores(oof.y.values, P)
# ... review manual -> tabel Keep/Correct/Remove/Unresolved -> label_overrides = {row_id: label_baru}
evaluate_cv("D2", train_df, PREPROCESS[BASELINE_CODE], label_overrides=label_overrides, drop_ids=drop_ids)
evaluate_cv("D2_orig_vs_corrected_eval", train_df, PREPROCESS[BASELINE_CODE],
            label_overrides=label_overrides, eval_on_corrected=True)

# Anggota 4 — SHAP pada model final (di luar loop CV)
import shap
ens = make_ensemble_fit_predict(keep_models=True)
X_tr, y_tr, X_te, feats = PREPROCESS[BASELINE_CODE](train_df.drop(columns=["row_id"]), test_df.drop(columns=["row_id", TARGET]))
out = ens(X_tr, y_tr, X_te)
explainer = shap.KernelExplainer(out["predict_fn"], shap.kmeans(X_tr, 10))
```
**Catatan Cleanlab:** probabilitas out-of-fold dihitung dari seluruh data train, lalu fold yang sama dipakai menilai D2/D3, sehingga kenaikan CV pada
eksperimen yang membuang/mengoreksi baris cenderung **optimistis**. Tulis ini sebagai limitasi; angka paling jujur ada di evaluasi test akhir.

## D4. Instruksi reproduksi (untuk laporan / README)
1. Python ≥ 3.10; `pip install numpy pandas scikit-learn tensorflow imbalanced-learn cleanlab shap lime` (catat versi dari sel setup).
2. Taruh `heart_disease_cleveland.csv` di `./dataset/`.
3. Hapus folder `./artifacts/` hanya jika ingin membuat split/fold baru; jika tidak, pertahankan agar semua anggota memakai split yang sama.
4. Jalankan notebook dari atas ke bawah (`Run All`). Seed global = 42; hasil TensorFlow bisa sedikit berbeda antar-mesin (±0,01 pada CV) meski seed sama.
5. Evaluasi test (`final_test_eval`) dijalankan **terakhir** dan masing-masing konfigurasi hanya sekali.